# IMF WEO Data Extraction

**Purpose:** Reshape the IMF World Economic Outlook Excel file from wide-by-year format  
into panel format suitable for the sovereign risk pipeline.

**Input:** `data/raw/world_economic_outlook_imf.xls`  
**Output:** `data/raw/weo_imf.csv`

**Source format (wide):**  
Each row = one country × one WEO variable. Year columns span 1980–2024.

| ISO | WEO Subject Code | Subject Descriptor | 1980 | 1981 | … | 2024 |
|-----|------------------|--------------------|------|------|---|------|
| USA | NGDP_R           | GDP, constant prices | 5123 | 5287 | … | 9876 |

**Target format (panel):**  
Each row = one country × one year. Each WEO variable becomes a column.

| iso | year | GDP, constant prices | Inflation, average … | … |
|-----|------|----------------------|----------------------|---|
| USA | 1980 | 5123                 | 13.5                 | … |

## Cell 1 — Imports and project root

In [ ]:
import sys
import logging
from pathlib import Path

import pandas as pd
import numpy as np


def find_project_root(start: Path = Path().resolve()) -> Path:
    """Walk up directory tree until the config/ folder is found."""
    for directory in [start, *start.parents]:
        if (directory / "config").is_dir():
            return directory
    raise FileNotFoundError(
        "Cannot find project root. "
        "Expected a config/ folder somewhere above this notebook."
    )


PROJECT_ROOT = find_project_root(Path("__file__").resolve().parent if "__file__" in dir() else Path().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {sys.version.split()[0]}")

## Cell 2 — Config and logging

In [ ]:
from config.settings import RAW_DATA_DIR, LOG_FORMAT, LOG_DATE_FORMAT

logging.basicConfig(
    level=logging.INFO,
    format=LOG_FORMAT,
    datefmt=LOG_DATE_FORMAT,
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("weo_extraction")

INPUT_FILE  = RAW_DATA_DIR / "world_economic_outlook_imf.xls"
OUTPUT_FILE = RAW_DATA_DIR / "weo_imf.csv"

log.info("Input  : %s", INPUT_FILE)
log.info("Output : %s", OUTPUT_FILE)
log.info("Exists : %s", INPUT_FILE.exists())

## Cell 3 — Load the WEO Excel file

The IMF WEO workbook typically contains one data sheet.  
We load it, inspect its shape, and identify the key structural columns.

In [ ]:
# IMF WEO .xls files use the legacy Excel format — requires xlrd.
# If xlrd is missing, install it: %pip install xlrd
try:
    import xlrd  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xlrd"])

# Inspect available sheets first
xf = pd.ExcelFile(INPUT_FILE, engine="xlrd")
log.info("Sheets: %s", xf.sheet_names)

# Load the first sheet (WEO data is always on sheet 0 / named 'WEO'
# or similar — adjust sheet_name if your file differs)
raw = pd.read_excel(
    INPUT_FILE,
    sheet_name=0,
    engine="xlrd",
    dtype=str,          # load everything as strings; we parse numerics later
    na_values=["", "n/a", "--", "...", "NA"],
)

log.info("Loaded  : %d rows × %d columns", *raw.shape)
print("\nFirst 5 rows (first 10 columns):")
print(raw.iloc[:5, :10].to_string())
print("\nAll column names:")
print(raw.columns.tolist())

## Cell 4 — Identify structural vs year columns

WEO files mix metadata columns (ISO code, Subject Descriptor, Units, etc.)  
with numeric year columns (1980 … 2024). We split these two groups here.

In [ ]:
# Key identifier columns — these names appear in every WEO release.
# Adjust the mapping below if your file uses slightly different header text.
COL_MAP = {
    "ISO":                "iso",
    "WEO Subject Code":   "weo_subject_code",
    "Subject Descriptor": "subject_descriptor",
    "Units":              "units",
    "Scale":              "scale",
}

# Verify all expected columns exist
missing_cols = [c for c in COL_MAP if c not in raw.columns]
if missing_cols:
    log.warning("These expected columns are NOT in the file: %s", missing_cols)
    log.warning("Available columns: %s", raw.columns.tolist())
else:
    log.info("All key identifier columns found.")

# Identify year columns: columns whose names are 4-digit integers in 1900-2100
def is_year_col(col_name: str) -> bool:
    try:
        yr = int(str(col_name).strip())
        return 1900 <= yr <= 2100
    except (ValueError, TypeError):
        return False

year_cols = [c for c in raw.columns if is_year_col(c)]
meta_cols = [c for c in raw.columns if not is_year_col(c)]

log.info("Year columns : %d  (%s … %s)", len(year_cols), year_cols[0], year_cols[-1])
log.info("Meta columns : %d  %s", len(meta_cols), meta_cols)

## Cell 5 — Drop trailing metadata and clean values

The WEO workbook often has footer rows (source notes, disclaimers) and an  
`Estimates Start After` column that we don't need for the panel. We drop those  
and clean numeric values (commas in thousands, stray whitespace).

In [ ]:
# Keep only rows that have a valid ISO code (drops footer/note rows)
df = raw.copy()
df = df[df["ISO"].notna()].copy()

# Drop housekeeping columns we don't need in the panel
DROP_COLS = [
    "WEO Country Code",
    "Country",
    "Subject Notes",
    "Country/Series-specific Notes",
    "Estimates Start After",
]
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

# Rename identifier columns to snake_case
df = df.rename(columns={k: v for k, v in COL_MAP.items() if k in df.columns})

# Clean numeric year values: strip whitespace, remove thousands commas
for col in year_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.replace(",", "", regex=False)  # "1,234.5" -> "1234.5"
        .replace({"nan": np.nan, "n/a": np.nan, "--": np.nan, "...": np.nan, "": np.nan})
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

log.info("After cleaning: %d rows × %d columns", *df.shape)
log.info("Unique countries (ISO): %d", df["iso"].nunique())
log.info("Unique variables      : %d", df["weo_subject_code"].nunique())
print("\nSample rows:")
print(df[["iso", "weo_subject_code", "subject_descriptor"] + year_cols[:5]].head(8).to_string(index=False))

## Cell 6 — Reshape: wide → long → panel

**Step 1 (melt):** Unpivot year columns → one row per (ISO, variable, year).  
**Step 2 (pivot_table):** Pivot Subject Descriptor → columns, giving one row per (ISO, year).  

We use `subject_descriptor` as column names because they are human-readable.  
The `weo_subject_code` → `subject_descriptor` mapping is printed at the end for reference.

In [ ]:
# ── Step 1: melt year columns to long format ──────────────────────────────────
id_vars = [c for c in ["iso", "weo_subject_code", "subject_descriptor", "units", "scale"]
           if c in df.columns]

long = df.melt(
    id_vars=id_vars,
    value_vars=year_cols,
    var_name="year",
    value_name="value",
)
long["year"] = long["year"].astype(int)

log.info("Long format  : %d rows × %d columns", *long.shape)

# ── Step 2: pivot subject_descriptor → columns ────────────────────────────────
# aggfunc='first' handles the rare case of duplicate (iso, year, subject) rows
panel = long.pivot_table(
    index=["iso", "year"],
    columns="subject_descriptor",
    values="value",
    aggfunc="first",
).reset_index()

# Flatten the MultiIndex column that pivot_table produces
panel.columns.name = None

# Sort by country then year
panel = panel.sort_values(["iso", "year"]).reset_index(drop=True)

log.info("Panel format : %d rows × %d columns", *panel.shape)
log.info("Countries    : %d", panel["iso"].nunique())
log.info("Years        : %d → %d", panel["year"].min(), panel["year"].max())
print("\nPanel columns:")
for i, col in enumerate(panel.columns):
    print(f"  [{i:3d}] {col}")

## Cell 7 — WEO subject code reference table

Print the mapping between `WEO Subject Code` and `Subject Descriptor`  
so it's easy to look up which code corresponds to which column.

In [ ]:
ref_cols = [c for c in ["weo_subject_code", "subject_descriptor", "units", "scale"] if c in df.columns]
reference = (
    df[ref_cols]
    .drop_duplicates(subset=["weo_subject_code"])
    .sort_values("weo_subject_code")
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 60)
print(f"Total WEO variables: {len(reference)}\n")
print(reference.to_string(index=False))

## Cell 8 — Data quality check

Completeness by column — flagged as OK / SPARSE / VERY SPARSE  
using the same thresholds as the World Bank notebook.

In [ ]:
variable_cols = [c for c in panel.columns if c not in ("iso", "year")]
total_rows    = len(panel)

print(f"{'Column':<55} {'Non-null':>9} {'Complete':>9}  Status")
print("-" * 90)

for col in variable_cols:
    non_null     = panel[col].notna().sum()
    completeness = non_null / total_rows * 100
    if completeness >= 80:
        status = "OK"
    elif completeness >= 50:
        status = "SPARSE"
    else:
        status = "VERY SPARSE"
    print(f"{col:<55} {non_null:>9,} {completeness:>8.1f}%  {status}")

print("-" * 90)
print(f"{'Total rows':<55} {total_rows:>9,}")

## Cell 9 — Preview sample countries

In [ ]:
preview_countries = ["USA", "BRA", "NGA"]
preview_years     = range(2018, 2025)   # last 6 years

mask = panel["iso"].isin(preview_countries) & panel["year"].isin(preview_years)
preview_cols = ["iso", "year"] + variable_cols[:6]   # first 6 variables

pd.set_option("display.max_columns", None)
pd.set_option("display.width",       200)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

print(panel.loc[mask, preview_cols].sort_values(["iso", "year"]).to_string(index=False))

## Cell 10 — Save to CSV

In [ ]:
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
panel.to_csv(OUTPUT_FILE, index=False)

size_kb = OUTPUT_FILE.stat().st_size / 1024

log.info("Saved      : %s", OUTPUT_FILE)
log.info("Rows       : %d", len(panel))
log.info("Columns    : %d  (iso, year + %d variables)", len(panel.columns), len(variable_cols))
log.info("File size  : %.1f KB", size_kb)
log.info("Countries  : %d  |  Years: %d–%d",
         panel["iso"].nunique(),
         panel["year"].min(),
         panel["year"].max())
log.info("")
log.info("Next steps:")
log.info("  1. Join weo_imf.csv with world_bank_data.csv on (iso / country, year).")
log.info("  2. Select the WEO variables relevant to the sovereign risk model.")
log.info("  3. Populate COUNTRIES_BY_TYPE in config/settings.py and run extract.ipynb.")